# Background Research and Measuring Relationship

This notebook will focus on measuring the relationship bewteen the proposed signals as well as some background research and examining the dataset. 

In [1]:
import os
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from   matplotlib.ticker import FuncFormatter

In [2]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path, ".."))
data_path = os.path.join(repo_path, "data")

In [3]:
tick_path = os.path.join(data_path, "InflationTickerGuide.xlsx")
df_ticker = pd.read_excel(io = tick_path, sheet_name = "FRED")

In [18]:
fred_namer = (df_ticker
    .set_index("ticker")
    .name
    .to_dict())

inf_namer = (pd
    .read_excel(io = tick_path, sheet_name = "InflationMeasures")
    .set_index("Name")
    .Description
    .to_dict())

RENAMER = {**fred_namer, **inf_namer}

# Examining Dataset

Below is table of the start and the end date of the FRED data

In [19]:
path = os.path.join(data_path, "FRED", "CombinedData.parquet")
display(pd
    .read_parquet(path = path, engine = "pyarrow")
    .drop(columns = ["group", "value"])
    .groupby("ticker")
    ["date"]
    .agg(["min", "max"])
    .reset_index()
    .assign(name = lambda x: x.ticker.map(RENAMER))
    .drop(columns = ["ticker"])
    .rename(columns = {
        "name": "Name",
        "min" : "Start Date",
        "max" : "End Date"})
    .set_index("Name"))

,Start Date,End Date
Name,,
Brent,1989-01-03,2026-07-27
WTI,1999-01-04,2026-07-27
Natural Gas,1997-01-07,2026-07-27
Heating Oil,1999-01-04,2026-07-27
RBOB,2003-03-11,2026-07-27
5y5y Forward Inflation,2003-01-02,2026-08-03


In [20]:
path = os.path.join(data_path, "PX", "FutPX.parquet")
display(pd
    .read_parquet(path = path, engine = "pyarrow")
    .assign(date = lambda x: pd.to_datetime(x.date).dt.date)
    .drop(columns = ["PX_LAST"])
    .groupby("security")
    ["date"]
    .agg(["min", "max"])
    .rename(columns = {
        "min": "Start Date",
        "max": "End Date"}))

,Start Date,End Date
security,,
CL1,1990-01-02,2024-10-29
CO1,1990-01-02,2024-10-29
HO1,1990-01-02,2024-10-29
NG1,1990-04-04,2024-10-29
QS1,1990-01-02,2024-10-29
XB1,2005-10-04,2024-10-29


In [30]:
path = os.path.join(data_path, "InflationMeasures")
display(pd
    .read_parquet(path = path, engine = "pyarrow")
    [["date", "security"]]
    .assign(date = lambda x: pd.to_datetime(x.date).dt.date)
    .replace(RENAMER)
    .groupby("security")
    ["date"]
    .agg(["min", "max"])
    .rename(columns = {
        "min": "Start Date",
        "max": "End Date"}))

,Start Date,End Date
security,,
GBP Inflation Swap Forward 5Y5Y,2004-04-27,2024-10-28
UK Bloomberg Economics Inflation Data Surprise,2001-01-01,2024-10-28
US Bloomberg Economics Inflation Data Surprise,2000-01-03,2024-10-28
USD Inflation Swap Forward 5Y5Y,2004-07-21,2024-10-28


# Replicating St. Louis Federal Reserve Graphs

In [ ]:
path   = os.path.join(data_path, "FRED", "CombinedData.parquet")
df_raw = pd.read_parquet(path = path, engine = "pyarrow")

In [ ]:
df_inf = (df_raw
    .loc[lambda x: x.group == "inflation"]
    .drop(columns = ["group", "ticker"])
    .rename(columns = {"value": "forward_inf"}))

In [ ]:
df_combined = (df_raw
    .loc[lambda x: x.group != "inflation"]
    .drop(columns = ["group"])
    .merge(right = df_inf, how = "inner", on = ["date"]))

In [ ]:
tickers   = df_combined.ticker.drop_duplicates().sort_values().to_list()
fig, axes = plt.subplots(ncols = 3, nrows = 2, figsize = (20,8))

for ticker, ax in zip(tickers, axes.flatten()):

    (df_combined
        .loc[lambda x: x.ticker == ticker]
        .plot(
            ax     = ax,
            kind   = "scatter", 
            x      = "forward_inf", 
            y      = "value",
            ylabel = "Spot Price ($)",
            xlabel = "5y5y Forward Inflation (%)",
            title  = RENAMER[ticker]))

    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.1f}%"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))

fig.suptitle("Comparing Spot Price vs. 5y5y Forward Inflation")
plt.tight_layout()